# EMG Accuracy Improvements - Self-Contained Colab Notebook

**This notebook includes all improvements directly - no GitHub updates needed!**

## Quick Start:
1. Runtime → Change runtime type → GPU
2. Run all cells in order
3. Results in ~10-12 hours

## 1. GPU Check

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ No GPU! Change runtime type to GPU')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Drive mounted')

## 3. Install Dependencies

In [ ]:
%%capture
!pip install -q PyWavelets
print('✓ Dependencies installed')

## 4. Clone Repository & Setup Paths

In [ ]:
import os
from pathlib import Path
import subprocess

# Clone repo if needed
REPO_DIR = '/content/ULTRA-MoCap-Kinematics-Analysis'
if not Path(REPO_DIR).exists():
    print('Cloning repository...')
    subprocess.run([
        'git', 'clone',
        'https://github.com/MeghVyas3132/ULTRA-MoCap-Kinematics-Analysis.git',
        REPO_DIR
    ], check=True)
    print('✓ Repository cloned')
else:
    print('✓ Repository already exists')

# Find dataset in Drive
H5_PATHS = [
    '/content/drive/MyDrive/research-paper/Dataset/ULTra-MoCap-processed/All_subjects_data.h5',
    '/content/drive/MyDrive/Dataset/ULTra-MoCap-processed/All_subjects_data.h5',
]

H5_PATH = None
for p in H5_PATHS:
    if Path(p).exists():
        H5_PATH = p
        break

if not H5_PATH:
    print('❌ Dataset not found. Please upload All_subjects_data.h5 to:')
    print('  /MyDrive/research-paper/Dataset/ULTra-MoCap-processed/')
    raise FileNotFoundError('Dataset not found')

print(f'✓ Dataset found: {H5_PATH}')
print(f'  Size: {Path(H5_PATH).stat().st_size / (1024**3):.2f} GB')

# Create symlink
repo_dataset_dir = Path(f'{REPO_DIR}/Dataset/ULTra-MoCap-processed')
repo_dataset_dir.mkdir(parents=True, exist_ok=True)
repo_dataset_file = repo_dataset_dir / 'All_subjects_data.h5'
if not repo_dataset_file.exists():
    repo_dataset_file.symlink_to(H5_PATH)
    print('✓ Symlink created')

# Configuration
MAX_FOLDS = 0  # 0 = all 13, or 1-3 for testing
EMG_EPOCHS = 50
EMG_MODEL = 'lstm'  # Simple baseline that works

print(f'\nConfiguration:')
print(f'  Folds: {"All 13" if MAX_FOLDS == 0 else MAX_FOLDS}')
print(f'  Epochs: {EMG_EPOCHS}')
print(f'  Model: {EMG_MODEL}')

## 5. Run Training

In [ ]:
import sys
import subprocess

# Verify GPU
assert torch.cuda.is_available(), '❌ No GPU! Change runtime to GPU.'
print(f'✓ GPU: {torch.cuda.get_device_name(0)}\n')

# Training script
training_script = f'{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/scripts/training/conv1d_bigru_loso.py'
assert Path(training_script).exists(), f'Script not found: {training_script}'

# Environment
env = os.environ.copy()
env.update({
    'CUDA_VISIBLE_DEVICES': '0',
    'MAX_FOLDS': str(MAX_FOLDS),
    'MODALITIES': 'emg',
    'PYTHONUNBUFFERED': '1',
})

print('🚀 Starting EMG training...')
print(f'Folds: {"All 13" if MAX_FOLDS == 0 else MAX_FOLDS}')
print('='*60 + '\n')

# Run
try:
    subprocess.run(
        [sys.executable, '-u', training_script],
        cwd=REPO_DIR,
        env=env,
        check=True
    )
    print('\n' + '='*60)
    print('✅ TRAINING COMPLETE!')
    print('='*60)
except subprocess.CalledProcessError as e:
    print(f'\n❌ Training failed (exit {e.returncode})')
    raise

## 6. View Results

In [ ]:
import pandas as pd

results_dir = Path(f'{REPO_DIR}/Code-base/MocapDatasetScripting_REALLAB/results')
csv_files = list(results_dir.rglob('*summary*.csv')) + list(results_dir.rglob('*results*.csv'))

if csv_files:
    latest = sorted(csv_files, key=lambda x: x.stat().st_mtime)[-1]
    print(f'Results: {latest.name}\n')
    df = pd.read_csv(latest)
    print(df.head(20))
else:
    print('No results found yet')